# Fine-Tuning YOLO11n VNetra: Tambah Kelas `vehicle`

Notebook ini menggabungkan dua dataset yang diupload secara manual:
1. **`vnetra_master_dataset`** — dataset navigasi lama (10 kelas, mAP@50 = 93.7%)
2. **`vehicle_dataset`** — dataset kendaraan baru (1 kelas: `vehicle`, nc=1)

Proses penggabungan dilakukan di dalam notebook ini dengan **remapping class ID**:
- Dataset navigasi: `pole=0, tactile_paving_straight=1, ..., tree=9` → digeser **+1** menjadi `1-10`
- Dataset vehicle: `vehicle=0` → tetap di posisi **0**
- Hasil akhir: 11 kelas, `vehicle=0, pole=1, ..., tree=10`

**Syarat Wajib Sebelum Run All:**
1. Upload `vnetra_master_dataset.zip` via **Add Data**
2. Upload `vehicle_dataset.zip` via **Add Data**
3. Upload `best.pt` dari training sebelumnya via **Add Data**

| Parameter | Training Pertama | Fine-Tuning Ini |
|---|---|---|
| Model Awal | `yolo11n.pt` (kosong) | `best.pt` (93.7% mAP) |
| Learning Rate | `lr0=0.002` | `lr0=0.0005` |
| Frozen Layers | `freeze=5` | `freeze=10` |
| Kelas | 10 navigasi | 11 (vehicle + navigasi) |


In [ ]:
!pip install -q albumentations
!pip install -q ultralytics


## 1. Persiapan Dataset: Gabung Navigasi + Vehicle

Cell ini menemukan kedua dataset, lalu menyalinnya ke `/kaggle/working` dengan **remapping class ID** yang aman:
- Semua label navigasi digeser `+1` (pole: 0→1, dst)
- Label vehicle tetap di ID=0
- Hasilnya: satu dataset 11 kelas siap training


In [ ]:
import os, shutil, yaml, glob

# ==========================================================
# LANGKAH 1: Temukan kedua dataset dari /kaggle/input
# ==========================================================
nav_dir = None
vehicle_dir = None

for root, dirs, files in os.walk('/kaggle/input'):
    if 'data.yaml' in files:
        with open(os.path.join(root, 'data.yaml'), 'r') as f:
            yml = yaml.safe_load(f)
        names = yml.get('names', [])
        if isinstance(names, dict): names = list(names.values())
        # Identifikasi berdasarkan nc dan nama kelas
        if yml.get('nc', 0) == 1 and 'vehicle' in names:
            vehicle_dir = root
            print(f'Dataset vehicle ditemukan: {root}')
        elif yml.get('nc', 0) >= 10 and 'pole' in names:
            nav_dir = root
            print(f'Dataset navigasi ditemukan: {root} ({yml["nc"]} kelas)')

if not nav_dir:
    raise FileNotFoundError('Dataset navigasi (vnetra_master_dataset) tidak ditemukan!')
if not vehicle_dir:
    raise FileNotFoundError('Dataset vehicle tidak ditemukan! Upload vehicle_dataset.zip.')

# ==========================================================
# LANGKAH 2: Definisi class ordering final (11 kelas)
# vehicle selalu di ID=0, navigasi digeser +1
# ==========================================================
master_classes = [
    'vehicle',                                                            # ID 0 (baru)
    'pole',                                                               # ID 1 (lama: 0+1)
    'tactile_paving_straight', 'tactile_paving_turn',                    # ID 2,3
    'tactile_paving_3way', 'tactile_paving_4way', 'tactile_paving_stop', # ID 4,5,6
    'stairs_up', 'stairs_down',                                          # ID 7,8
    'crosswalk', 'tree'                                                   # ID 9,10
]
print(f'\nKelas final ({len(master_classes)} kelas):')
for idx, cls in enumerate(master_classes):
    print(f'  ID {idx}: {cls}')

# ==========================================================
# LANGKAH 3: Buat folder master gabungan
# ==========================================================
master_dir = '/kaggle/working/vnetra_master_dataset_vehicle'
if os.path.exists(master_dir): shutil.rmtree(master_dir)
for split in ['train', 'valid', 'test']:
    os.makedirs(f'{master_dir}/{split}/images', exist_ok=True)
    os.makedirs(f'{master_dir}/{split}/labels', exist_ok=True)

print(f'\nFolder master dibuat: {master_dir}')

# ==========================================================
# LANGKAH 4: Salin dataset NAVIGASI dengan remap ID +1
# ==========================================================
print('\nMenyalin dataset navigasi (remap IDs +1)...')
nav_copied = 0
for split in ['train', 'valid', 'test']:
    src_img = os.path.join(nav_dir, split, 'images')
    src_lbl = os.path.join(nav_dir, split, 'labels')
    dst_img = f'{master_dir}/{split}/images'
    dst_lbl = f'{master_dir}/{split}/labels'

    if not os.path.exists(src_img): continue

    for fname in os.listdir(src_img):
        if not fname.endswith(('.jpg', '.jpeg', '.png')): continue
        lbl_name = fname.rsplit('.', 1)[0] + '.txt'
        lbl_src_path = os.path.join(src_lbl, lbl_name)
        if not os.path.exists(lbl_src_path): continue

        # REMAP: baca label lama, tambahkan +1 pada setiap class ID
        new_lines = []
        with open(lbl_src_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if not parts: continue
                old_id = int(parts[0])
                new_id = old_id + 1  # Geser +1 karena vehicle masuk di posisi 0
                new_lines.append(f"{new_id} {' '.join(parts[1:])}\n")

        if not new_lines: continue

        # Salin gambar dan tulis label baru
        shutil.copy(os.path.join(src_img, fname), os.path.join(dst_img, fname))
        with open(os.path.join(dst_lbl, lbl_name), 'w') as f:
            f.writelines(new_lines)
        nav_copied += 1

print(f'  Navigasi: {nav_copied} gambar disalin (IDs digeser +1)')

# ==========================================================
# LANGKAH 5: Salin dataset VEHICLE (ID=0, tidak perlu remap)
# ==========================================================
print('\nMenyalin dataset vehicle (ID=0, tidak perlu remap)...')
veh_copied = 0
for split in ['train', 'valid', 'test']:
    src_img = os.path.join(vehicle_dir, split, 'images')
    src_lbl = os.path.join(vehicle_dir, split, 'labels')
    dst_img = f'{master_dir}/{split}/images'
    dst_lbl = f'{master_dir}/{split}/labels'

    if not os.path.exists(src_img): continue

    for fname in os.listdir(src_img):
        if not fname.endswith(('.jpg', '.jpeg', '.png')): continue
        lbl_name = fname.rsplit('.', 1)[0] + '.txt'
        lbl_src_path = os.path.join(src_lbl, lbl_name)
        if not os.path.exists(lbl_src_path): continue

        # Cegah nama file duplikat: tambahkan prefix 'veh_'
        dst_fname = 'veh_' + fname
        dst_lbl_name = 'veh_' + lbl_name

        dst_img_path = os.path.join(dst_img, dst_fname)
        dst_lbl_path = os.path.join(dst_lbl, dst_lbl_name)

        if os.path.exists(dst_img_path): continue  # Skip jika sudah ada

        # Vehicle label: ID=0 (vehicle), langsung salin tanpa remap
        shutil.copy(os.path.join(src_img, fname), dst_img_path)
        shutil.copy(lbl_src_path, dst_lbl_path)
        veh_copied += 1

print(f'  Vehicle: {veh_copied} gambar disalin (ID=0, no remap)')

# ==========================================================
# LANGKAH 6: Tulis data.yaml untuk dataset gabungan 11 kelas
# ==========================================================
with open(f'{master_dir}/data.yaml', 'w') as f:
    yaml.dump({
        'path': master_dir,
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': len(master_classes),
        'names': master_classes
    }, f, sort_keys=False, allow_unicode=True)

print(f'\ndata.yaml ditulis: {len(master_classes)} kelas')
print(f'Master dir siap: {master_dir}')


## 2. Laporan Proporsi Dataset Gabungan
Verifikasi distribusi kelas setelah penggabungan. Kelas `vehicle` seharusnya muncul di semua split.


In [ ]:
import pandas as pd
import os, yaml

# Reload master_classes dari data.yaml (dinamis, tidak hardcode)
with open(f'{master_dir}/data.yaml', 'r') as f:
    _yaml = yaml.safe_load(f)
    master_classes = _yaml.get('names', [])

def count_images(directory):
    if not os.path.exists(directory): return 0
    return len([f for f in os.listdir(directory) if f.endswith(('.jpg', '.jpeg', '.png'))])

train_count = count_images(f'{master_dir}/train/images')
valid_count = count_images(f'{master_dir}/valid/images')
test_count  = count_images(f'{master_dir}/test/images')
total_images = train_count + valid_count + test_count

print('=== Statistik Keseluruhan ===')
print(f'Total Gambar      : {total_images}')
print(f'Train             : {train_count}')
print(f'Valid             : {valid_count}')
print(f'Test              : {test_count}')
print()

def count_instances_per_class(label_dir, num_classes):
    counts = {i: 0 for i in range(num_classes)}
    if not os.path.exists(label_dir): return counts
    for lbl_file in os.listdir(label_dir):
        if not lbl_file.endswith('.txt'): continue
        with open(os.path.join(label_dir, lbl_file), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cid = int(parts[0])
                    if cid in counts: counts[cid] += 1
    return counts

train_cls = count_instances_per_class(f'{master_dir}/train/labels', len(master_classes))
valid_cls = count_instances_per_class(f'{master_dir}/valid/labels', len(master_classes))
test_cls  = count_instances_per_class(f'{master_dir}/test/labels', len(master_classes))

data_report = []
total_train = total_valid = total_test = global_total = 0
for i, cls_name in enumerate(master_classes):
    t_train, t_valid, t_test = train_cls[i], valid_cls[i], test_cls[i]
    t_total = t_train + t_valid + t_test
    total_train += t_train; total_valid += t_valid
    total_test += t_test; global_total += t_total
    data_report.append({'ID': i, 'Kelas': cls_name,
                        'Train (Inst)': t_train, 'Valid (Inst)': t_valid,
                        'Test (Inst)': t_test, 'Total Instance': t_total})
data_report.append({'ID': '-', 'Kelas': 'TOTAL', 'Train (Inst)': total_train,
                    'Valid (Inst)': total_valid, 'Test (Inst)': total_test,
                    'Total Instance': global_total})

import pandas as pd
display(pd.DataFrame(data_report))


## 3. Fine-Tuning YOLO11n dari `best.pt`

Konfigurasi khusus **Incremental Learning** untuk mencegah *Catastrophic Forgetting*:
- **`model = YOLO('best.pt')`** — mulai dari model terbaik sebelumnya, bukan dari awal
- **`lr0=0.0005`** — learning rate sangat kecil agar tidak merusak pengetahuan kelas lama
- **`freeze=10`** — bekukan lebih banyak layer backbone agar fitur dasar terjaga
- **`patience=15`** — lebih cepat berhenti jika model sudah konvergen


In [ ]:
from ultralytics import YOLO
import os

# Path dataset vehicle (disesuaikan dari Cell 3)
master_dir = '/kaggle/working/vnetra_master_dataset_vehicle'

# ==========================================================
# SMART AUTO-DETECT: Cari best.pt dari semua sumber
# Mendukung 3 cara Add Data di Kaggle:
#   1. Notebook Output  -> /kaggle/input/notebooks/<user>/<slug>/.../best.pt
#   2. Dataset Upload   -> /kaggle/input/<dataset-slug>/best.pt
#   3. Working dir      -> /kaggle/working/best.pt (fallback)
# ==========================================================
def find_best_pt():
    candidates = []  # list of (score, path)

    for search_dir in ['/kaggle/input', '/kaggle/working']:
        for root, dirs, files in os.walk(search_dir):
            for fname in files:
                # Cari file .pt yang berisi bobot YOLO VNetra
                # Nama kandidat yang diterima: best.pt atau best_yolo11n.pt
                if fname not in ('best.pt', 'best_yolo11n.pt'):
                    continue

                full_path = os.path.join(root, fname)
                score = 0

                # Prioritas tinggi: dari output notebook Kaggle (ada '/notebooks/')
                if '/notebooks/' in root:
                    score += 100

                # Prioritas lebih tinggi: dari folder training VNetra yang dikenal
                if 'vnetra_training' in root:
                    score += 50
                if 'yolo11n_custom' in root:
                    score += 30

                # Prioritas lebih tinggi: ada di folder weights/ (bukan sembarang .pt)
                if 'weights' in root:
                    score += 20

                # Nama file best.pt lebih disukai daripada nama lain
                if fname == 'best.pt':
                    score += 10

                # Dari /kaggle/input lebih dipercaya daripada /kaggle/working
                if root.startswith('/kaggle/input'):
                    score += 5

                candidates.append((score, full_path))
                print(f'  [score={score:3d}] {full_path}')

    if not candidates:
        return None

    # Pilih yang paling tinggi skor-nya
    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1]


print('Mencari best.pt dari semua sumber di /kaggle/input...')
best_pt_path = find_best_pt()

if not best_pt_path:
    raise FileNotFoundError(
        '\nbest.pt tidak ditemukan! Pastikan salah satu dari:\n'
        '  1. Tambahkan Output Notebook training pertama via Add Data (Notebooks tab)\n'
        '  2. Upload best.pt sebagai Dataset di Kaggle\n'
        '  3. Salin best.pt ke /kaggle/working/ secara manual'
    )

print(f'\nDipilih: {best_pt_path}')

model = YOLO(best_pt_path)
print(f'Model berhasil dimuat!')
print(f'Jumlah kelas pada model lama: {model.model.nc}')

    # --- [OPSIONAL] ALARM REM DARURAT SISA KUOTA ---
    # import time
    # def alarm_kuota(trainer):
    #     if time.time() - trainer.train_time_start > 5400:
    #         print('ALARM: Sisa kuota hampir habis! Menyimpan progress...')
    #         trainer.stop = True
    # model.add_callback('on_train_epoch_end', alarm_kuota)
    # -----------------------------------------------

results = model.train(
    # --- KONFIGURASI DATA & PERANGKAT ---
    data=f'{master_dir}/data.yaml',  # data.yaml berisi 11 kelas (10 lama + vehicle)
    epochs=150,                      # Cukup 150 epoch untuk fine-tuning (tidak perlu dari 0)
    time=10.0,                       # Otomatis berhenti setelah 10 jam (aman dari Kaggle timeout 12 jam)
    patience=15,                     # Lebih cepat berhenti — model lama sudah pintar, tidak perlu banyak iterasi
    imgsz=640,                       # Resolusi identik dengan training pertama
    batch=128,                       # Memanfaatkan 2x GPU T4 Kaggle
    device=[0, 1],                   # Multi-GPU T4
    workers=8,
    seed=42,

    # --- PENYIMPANAN LOG & GRAFIK ---
    project='vnetra_training',
    name='yolo11n_vehicle_finetune',  # Nama berbeda agar tidak menimpa training pertama
    exist_ok=True,
    save_period=10,                  # Lebih sering save — fine-tuning konvergen lebih cepat

    # ============================================================
    # STRATEGI ANTI CATASTROPHIC FORGETTING
    # (Kunci agar kelas lama tidak 'dilupakan' saat belajar vehicle)
    # ============================================================
    freeze=10,                       # 2x lebih banyak frozen layer dari training pertama (freeze=5)
                                     # Layer 0-10 tidak berubah: fitur dasar tepi/warna terjaga
    optimizer='AdamW',               # Konsisten dengan training pertama
    lr0=0.0005,                      # 4x lebih kecil dari training pertama (0.002)
                                     # Krusial: perubahan bobot sangat halus, tidak merusak kelas lama
    cos_lr=True,                     # Penurunan lr membentuk kurva kosinus
    warmup_epochs=3.0,               # Lebih singkat — model sudah 'hangat'

    # --- RASIO KOMPROMI LOKALISASI (sama dengan training pertama) ---
    box=7.5,
    cls=1.5,
    dfl=1.5,

    # --- AUGMENTASI KHUSUS VNETRA (OV2640 CAMERA SIMULATION) ---
    mosaic=1.0,
    degrees=8.0,
    fliplr=1.0,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    erasing=0.1,
)


In [ ]:
import shutil
import os

# Salin model terbaru (sudah termasuk kelas vehicle) ke working dir
src_pt = 'runs/detect/vnetra_training/yolo11n_vehicle_finetune/weights/best.pt'
shutil.copy(src_pt, '/kaggle/working/best_vehicle_yolo11n.pt')
print('Model Fine-Tuned (.pt) siap di-download!')


## 5. Export ke LiteRT (FP32 & INT8)
Mengekspor bobot model menjadi format `.tflite` dalam dua bentuk kuantisasi:
1. **FP32** (Half Precision) -> Sangat efisien dan kompatibel untuk *GPU Delegation* di Android.
2. **INT8** (Full Integer) -> Wajib untuk akselerator *NPU / NNAPI* yang membutuhkan model super ringan.

In [ ]:

print("Mengekspor model ke FP32...")
# 1. Export ke TFLite (FP32) - Optimal untuk GPU Mobile (Proses cepat)
export_fp32 = model.export(format="litert", optimize=True)
print("===========================================================")
print("Export FP32 Selesai! Lokasi file TFLite:")
print("FP32:", export_fp32)
print("===========================================================")

print("===========================================================")
print("Export INT8 Selesai! Lokasi file TFLite:")
print("===========================================================")


print("Mengekspor model ke INT8 (TFLite Quantization)...")
# Export INT8 memerlukan data.yaml untuk proses kalibrasi (menghitung rentang aktivasi)
export_int8 = model.export(format="litert", optimize=True, int8=True, data=f"{master_dir}/data.yaml")
print("===========================================================")
print("Export INT8 Selesai! Lokasi file TFLite:")
print("INT8:", export_int8)
print("===========================================================")


In [ ]:
import shutil
import os
shutil.copy(export_fp32, '/kaggle/working/best_fp32.tflite')
shutil.copy(export_int8, '/kaggle/working/best_int8.tflite')
print('Model FP32 dan INT8 siap di-download!')


## 6. Validasi Kuantisasi (Benchmarking Skripsi)
Menguji kembali model pada Test Set untuk melihat seberapa jauh penurunan akurasi (mAP) akibat proses kompresi FP32 dibanding model aslinya.

In [ ]:
import gc
gc.collect()

print("\n=== EVALUASI MODEL ASLI (.pt) PADA TEST SET ===")
val_pt = model.val(data=f"{master_dir}/data.yaml", split='test')
map_pt = val_pt.box.map50

print("\n=========================================")
print("mAP@50 (Akurasi):")
print(f"Original (.pt)   : {map_pt:.4f}")


In [ ]:
print("\n=== EVALUASI MODEL FP32 (.tflite) PADA TEST SET ===")
model_fp32 = YOLO(export_fp32, task='detect')
val_fp32 = model_fp32.val(data=f"{master_dir}/data.yaml", split='test')
map_fp32 = val_fp32.box.map50

print("=== EVALUASI MODEL INT8 (.tflite) PADA TEST SET ===")
model_int8 = YOLO(export_int8, task='detect')
val_int8 = model_int8.val(data=f"{master_dir}/data.yaml", split='test')
map_int8 = val_int8.box.map50

print("=========================================")
print("mAP@50 (Akurasi):")
print(f"FP32 (.tflite)   : {map_fp32:.4f}")
print(f"INT8 (.tflite)   : {map_int8:.4f}")



In [ ]:
print("=========================================")
print("KESIMPULAN PERBANDINGAN mAP@50 PADA DATASET TEST:")
print(f"Original (.pt)   : {map_pt:.4f}")
print(f"FP32 (.tflite)   : {map_fp32:.4f}")
print(f"INT8 (.tflite)   : {map_int8:.4f}")
print("=========================================")


## 7. Pengujian Visualisasi Langsung (Predict)
Mengambil satu gambar tes secara acak dan menampilkan prediksi kotak deteksi dari model asli (.pt) vs model terkompresi (.tflite) agar Anda bisa meletakkannya di Laporan Skripsi.

In [ ]:
import random
import matplotlib.pyplot as plt
import cv2
import glob

# Pilih satu gambar acak dari dataset test
test_images = glob.glob(f"{master_dir}/test/images/*.jpg")
if test_images:
    test_img = random.choice(test_images)
    print(f"Menguji gambar: {test_img}")
    
    # Prediksi pakai model Asli
    res_pt = model.predict(source=test_img, imgsz=640)
    img_pt = res_pt[0].plot()
    
    # Prediksi pakai model FP32
    res_fp32 = model_fp32.predict(source=test_img, imgsz=640)
    img_fp32 = res_fp32[0].plot()
    
    # Prediksi pakai model INT8
    res_int8 = model_int8.predict(source=test_img, imgsz=640)
    img_int8 = res_int8[0].plot()
    
    # Tampilkan perbandingan
    fig, ax = plt.subplots(1, 3, figsize=(20, 7))
    ax[0].imshow(cv2.cvtColor(img_pt, cv2.COLOR_BGR2RGB))
    ax[0].set_title("Prediksi Model Asli (.pt)")
    ax[0].axis("off")
    
    ax[1].imshow(cv2.cvtColor(img_fp32, cv2.COLOR_BGR2RGB))
    ax[1].set_title("Prediksi Model FP32 (.tflite)")
    ax[1].axis("off")
    
    ax[2].imshow(cv2.cvtColor(img_int8, cv2.COLOR_BGR2RGB))
    ax[2].set_title("Prediksi Model INT8 (.tflite)")
    ax[2].axis("off")
    
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada gambar di folder test untuk diprediksi.")


## 8. Visualisasi Grafik Hasil Training
Menampilkan grafik metrik akurasi (*mAP*, *Loss*) dan *Confusion Matrix* yang telah digenerasi oleh YOLO menggunakan `matplotlib` untuk keperluan laporan skripsi.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

base_path = '/kaggle/working/runs/detect/vnetra_training/yolo11n_vehicle_finetune/'
results_path = os.path.join(base_path, 'results.csv')

print('=== 1. CUSTOM TRAINING DASHBOARD (SEABORN) ===')
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    df.columns = df.columns.str.strip()  # Bersihkan spasi berlebih di nama kolom
    
    sns.set_theme(style='whitegrid', palette='deep')
    fig, axes = plt.subplots(2, 2, figsize=(20, 14))
    fig.suptitle('VNetra - YOLO11n Training Performance Dashboard', fontsize=26, fontweight='bold', y=0.96)
    
    # 1. Box Loss Convergence
    sns.lineplot(data=df, x='epoch', y='train/box_loss', ax=axes[0,0], label='Train Box Loss', linewidth=3, color='#1f77b4')
    sns.lineplot(data=df, x='epoch', y='val/box_loss', ax=axes[0,0], label='Val Box Loss', linewidth=3, color='#ff7f0e', linestyle='--')
    axes[0,0].set_title('Box Loss Convergence', fontsize=18, fontweight='bold')
    axes[0,0].set_xlabel('Epoch', fontsize=14)
    axes[0,0].set_ylabel('Loss', fontsize=14)
    axes[0,0].legend(fontsize=12, frameon=True, shadow=True)
    
    # 2. Class Loss Convergence
    sns.lineplot(data=df, x='epoch', y='train/cls_loss', ax=axes[0,1], label='Train Class Loss', linewidth=3, color='#2ca02c')
    sns.lineplot(data=df, x='epoch', y='val/cls_loss', ax=axes[0,1], label='Val Class Loss', linewidth=3, color='#d62728', linestyle='--')
    axes[0,1].set_title('Classification Loss Convergence', fontsize=18, fontweight='bold')
    axes[0,1].set_xlabel('Epoch', fontsize=14)
    axes[0,1].set_ylabel('Loss', fontsize=14)
    axes[0,1].legend(fontsize=12, frameon=True, shadow=True)
    
    # 3. mAP Score Evolution
    sns.lineplot(data=df, x='epoch', y='metrics/mAP50(B)', ax=axes[1,0], label='mAP@50', linewidth=3, color='#9467bd')
    sns.lineplot(data=df, x='epoch', y='metrics/mAP50-95(B)', ax=axes[1,0], label='mAP@50-95', linewidth=3, color='#8c564b', linestyle='-.')
    axes[1,0].set_title('Mean Average Precision (mAP)', fontsize=18, fontweight='bold')
    axes[1,0].set_xlabel('Epoch', fontsize=14)
    axes[1,0].set_ylabel('Score', fontsize=14)
    axes[1,0].legend(fontsize=12, frameon=True, shadow=True)
    
    # 4. Precision & Recall
    sns.lineplot(data=df, x='epoch', y='metrics/precision(B)', ax=axes[1,1], label='Precision', linewidth=3, color='#e377c2')
    sns.lineplot(data=df, x='epoch', y='metrics/recall(B)', ax=axes[1,1], label='Recall', linewidth=3, color='#17becf', linestyle=':')
    axes[1,1].set_title('Precision & Recall Trends', fontsize=18, fontweight='bold')
    axes[1,1].set_xlabel('Epoch', fontsize=14)
    axes[1,1].set_ylabel('Score', fontsize=14)
    axes[1,1].legend(fontsize=12, frameon=True, shadow=True)
    
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()
else:
    print('File results.csv belum ditemukan.')

def display_result(image_path, width=None):
    if os.path.exists(image_path):
        if width:
            display(Image(filename=image_path, width=width))
        else:
            display(Image(filename=image_path))
    else:
        print(f'\u26a0\ufe0f Gambar tidak ditemukan: {os.path.basename(image_path)}')
        print('   (Gambar ini baru akan digenerate oleh YOLO di detik terakhir setelah epoch 50 selesai 100%)')

print('\n=== 2. CONFUSION MATRIX PROFESIONAL ===')
display_result(os.path.join(base_path, 'confusion_matrix_normalized.png'), width=1200)

print('\n=== 3. KURVA F1-SCORE (Confidence Thresholding) ===')
display_result(os.path.join(base_path, 'BoxF1_curve.png'), width=1200)

print('\n=== 4. VISUALISASI AUGMENTASI MOSAIC PADA DATA TRAINING ===')
display_result(os.path.join(base_path, 'train_batch0.jpg'), width=1200)

print('\n=== 5. SAMPEL PREDIKSI PADA VALIDATION SET ===')
display_result(os.path.join(base_path, 'val_batch0_pred.jpg'), width=1200)



## 9. Paketkan Hasil Training (ZIP)
Membungkus seluruh grafik evaluasi, kurva performa, matriks kebingungan (*confusion matrix*), dan file model bobot (`best.pt` & `best_fp32.tflite`) ke dalam satu file ZIP yang sangat praktis untuk Anda unduh (tanpa mengikutkan dataset gambar untuk menghemat kuota).

In [ ]:
import os
import zipfile

zip_path = "/kaggle/working/vnetra_training_results.zip"
# Path otomatis mendeteksi folder training YOLO
base_run_dir = "/kaggle/working/runs/detect/vnetra_training"

print("Membuat arsip ZIP untuk hasil training...")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    # Memasukkan folder YOLO training
    if os.path.exists(base_run_dir):
        for root, dirs, files in os.walk(base_run_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, "/kaggle/working")
                try:
                    zipf.write(file_path, arcname)
                except Exception as e:
                    print(f"Skip file {file} karena error: {e}")
    else:
        print("⚠️ Peringatan: Folder training YOLO tidak ditemukan!")
        
    # [NEW] Memastikan model TFLite ikut masuk ZIP (di root zip)
    tflite_files = ['/kaggle/working/best_fp32.tflite', '/kaggle/working/best_int8.tflite']
    for tflite in tflite_files:
        if os.path.exists(tflite):
            try:
                zipf.write(tflite, os.path.basename(tflite))
                print(f"Menambahkan {os.path.basename(tflite)} ke dalam ZIP")
            except Exception as e:
                print(f"Gagal menambahkan {tflite}: {e}")

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"✅ Berhasil! Silakan unduh file: {zip_path} ({size_mb:.2f} MB)")
    print("File ini berisi semua model terlatih (.pt & .tflite) beserta grafik evaluasi.")
else:
    print("❌ Gagal membuat file ZIP.")


## 10. Bersihkan Sisa Dataset (Opsional)
Menghapus folder dataset dari `/kaggle/working/` agar **tidak ikut tersimpan** menjadi Output Kaggle. Hal ini akan sangat menghemat kapasitas penyimpanan dan mempercepat proses *Save & Run All*.

In [ ]:
import shutil
import os

master_dir = '/kaggle/working/vnetra_master_dataset_vehicle'

if os.path.exists(master_dir):
    print("🗑️ Menghapus folder dataset dari working directory...")
    try:
        shutil.rmtree(master_dir)
        print("✅ Dataset berhasil dihapus. Output Kaggle Anda kini akan jauh lebih ringan!")
    except Exception as e:
        print(f"❌ Gagal menghapus dataset: {e}")
else:
    print("Dataset sudah tidak ada.")
